In [10]:
import os
import re
import math
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ==========================================================
# PATHS
# ==========================================================

INPUT_ROOT = r"D:\Malvika\fight clips\Crime_Project\Fight Frames"
OUTPUT_ROOT = r"D:\Malvika\fight clips\Crime_Project\Fight_Frames_Annotated"

ANNOTATION_FILES = [
    r"D:\Malvika\fight clips\Crime_Project\annotations\UCFCrime_Train.txt",
    r"D:\Malvika\fight clips\Crime_Project\annotations\UCFCrime_Val.txt",
    r"D:\Malvika\fight clips\Crime_Project\annotations\UCFCrime_Test.txt"
]

# ==========================================================
# SETTINGS
# ==========================================================

CONTEXT_SECONDS = 1          # Keep 1 second before & after annotation
IMAGE_EXTENSION = ".jpg"

REPORT_NAME = "reduction_report.csv"

In [18]:
def time_to_frame(time_string):
    """
    Converts timestamp to frame number.
    Supports:
        mm:ss.s
        hh:mm:ss.s

    Assumes extraction at 1 FPS.
    """

    parts = time_string.split(":")

    if len(parts) == 2:
        minutes = int(parts[0])
        seconds = float(parts[1])
        total_seconds = minutes * 60 + seconds

    elif len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = float(parts[2])
        total_seconds = hours * 3600 + minutes * 60 + seconds

    else:
        raise ValueError(f"Invalid timestamp: {time_string}")

    return int(round(total_seconds))

In [19]:
# ==========================================================
# Load Annotation Files
# ==========================================================

def load_annotations(annotation_files):

    annotation_dict = {}

    for annotation_file in annotation_files:

        print(f"Reading: {annotation_file}")

        with open(annotation_file, "r", encoding="utf-8") as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                # Ignore description after ##
                line = line.split("##")[0].strip()

                parts = line.split()

                if len(parts) < 3:
                    continue

                video_name = parts[0]
                start_time = parts[1]
                end_time = parts[2]

                start_frame = max(0, time_to_frame(start_time) - CONTEXT_SECONDS)
                end_frame = time_to_frame(end_time) + CONTEXT_SECONDS

                if video_name not in annotation_dict:
                    annotation_dict[video_name] = []

                annotation_dict[video_name].append((start_frame, end_frame))

    return annotation_dict

In [20]:
annotations = load_annotations(ANNOTATION_FILES)

print("\nTotal videos loaded :", len(annotations))

Reading: D:\Malvika\fight clips\Crime_Project\annotations\UCFCrime_Train.txt
Reading: D:\Malvika\fight clips\Crime_Project\annotations\UCFCrime_Val.txt
Reading: D:\Malvika\fight clips\Crime_Project\annotations\UCFCrime_Test.txt

Total videos loaded : 1854


In [21]:
print("Total annotated videos:", len(annotations))

Total annotated videos: 1854


In [22]:
print(annotations["Fighting002_x264"])

[(0, 3), (1, 7), (3, 7), (5, 9), (7, 12), (10, 24), (26, 38), (60, 75), (73, 82)]


In [23]:
def merge_intervals(intervals):
    """
    Merge overlapping intervals.
    """

    if not intervals:
        return []

    intervals = sorted(intervals)

    merged = [list(intervals[0])]

    for start, end in intervals[1:]:

        last_start, last_end = merged[-1]

        if start <= last_end:
            merged[-1][1] = max(last_end, end)
        else:
            merged.append([start, end])

    return [tuple(x) for x in merged]

In [24]:
video = "Fighting002_x264"

print("Before:")
print(annotations[video])

print()

merged = merge_intervals(annotations[video])

print("After:")
print(merged)

Before:
[(0, 3), (1, 7), (3, 7), (5, 9), (7, 12), (10, 24), (26, 38), (60, 75), (73, 82)]

After:
[(0, 24), (26, 38), (60, 82)]


In [25]:
# ==========================================================
# Copy selected frames for ONE video
# ==========================================================

def copy_selected_frames(video_folder, intervals, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    copied = 0
    total = 0

    # Get all jpg files
    frame_files = sorted([
        f for f in os.listdir(video_folder)
        if f.lower().endswith(IMAGE_EXTENSION)
    ])

    total = len(frame_files)

    for frame_file in frame_files:

        # Extract frame number from filename
        # Example:
        # Fighting002_x264_0008.jpg -> 8

        match = re.search(r'_(\d+)\.jpg$', frame_file)

        if match is None:
            continue

        frame_number = int(match.group(1))

        keep = False

        for start, end in intervals:

            if start <= frame_number <= end:
                keep = True
                break

        if keep:

            shutil.copy2(
                os.path.join(video_folder, frame_file),
                os.path.join(output_folder, frame_file)
            )

            copied += 1

    return total, copied

In [26]:
video = "Fighting002_x264"

video_folder = os.path.join(INPUT_ROOT, video)

output_folder = os.path.join(
    OUTPUT_ROOT,
    video
)

intervals = merge_intervals(annotations[video])

before, after = copy_selected_frames(
    video_folder,
    intervals,
    output_folder
)

print(f"Frames before : {before}")
print(f"Frames after  : {after}")

Frames before : 90
Frames after  : 61


In [27]:
# ==========================================================
# Process all videos
# ==========================================================

report = []

video_folders = sorted([
    f for f in os.listdir(INPUT_ROOT)
    if os.path.isdir(os.path.join(INPUT_ROOT, f))
])

print(f"Found {len(video_folders)} video folders.\n")

for video in tqdm(video_folders):

    # Skip videos without annotations
    if video not in annotations:
        print(f"Skipping (no annotation): {video}")
        continue

    video_folder = os.path.join(INPUT_ROOT, video)

    output_folder = os.path.join(
        OUTPUT_ROOT,
        video
    )

    intervals = merge_intervals(annotations[video])

    before, after = copy_selected_frames(
        video_folder,
        intervals,
        output_folder
    )

    reduction = round((1 - after / before) * 100, 2) if before else 0

    report.append({
        "Video": video,
        "Frames Before": before,
        "Frames After": after,
        "Reduction %": reduction,
        "Intervals": len(intervals)
    })

Found 25 video folders.



100%|██████████| 25/25 [00:02<00:00, 11.81it/s]


In [29]:
report_df = pd.DataFrame(report)


REPORT_NAME = os.path.join(
    OUTPUT_ROOT,
    "reduction_report.csv"
)

report_df.to_csv(REPORT_NAME, index=False)

print(report_df.head())

print()

print("Done!")

print(f"Videos processed : {len(report_df)}")
print(f"Report saved     : {REPORT_NAME}")

              Video  Frames Before  Frames After  Reduction %  Intervals
0  Fighting002_x264             90            61        32.22          3
1  Fighting003_x264            104            95         8.65          3
2  Fighting006_x264             32            30         6.25          1
3  Fighting007_x264            127           126         0.79          1
4  Fighting011_x264            275           236        14.18          8

Done!
Videos processed : 25
Report saved     : D:\Malvika\fight clips\Crime_Project\Fight_Frames_Annotated\reduction_report.csv
